# Fraud Detection: Model Comparison

Milestone 3. We compare four algorithms on our client table, all with **default settings**, and choose the one that goes on to **hyperparameter tuning in notebook 11**.

**Selection rule:** the model with the highest **mean ROC-AUC in 5-fold cross-validation** wins. The validation split is a second check.

**Metrics** (team decision, see [protocol.md](protocol.md)):
- **ROC-AUC** (main metric): how well the model ranks fraud cases above honest clients.
- **PR-AUC**: how clean the top of the ranking is; important because only 5.6% of the clients are fraud (random guessing = 0.056).
- **Precision, recall and F1** at the F1-best threshold: the business view. The threshold is chosen on the validation data, so these are an upper bound.

**Fair setup:**
- every model uses its default settings, no tuning yet,
- every model gives the fraud class more weight (class-weight decision in `protocol.md`),
- all models see the same clients: cross-validation on the training split, then one check on the validation split (20% of the Zindi train clients). The Zindi test set is not used here.

The notebook shows only tables and numbers, no charts.

In [1]:
# libraries for data handling, the four models, cross-validation and the metrics
import numpy as np
import pandas as pd
from pathlib import Path
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 40)
FEATURE_DIR = Path("data/features")
RANDOM_STATE = 42

## 1. Load the feature tables

From [08_feature-engineering.ipynb](08_feature-engineering.ipynb), index = `client_id`:
- **tree tables** (`X_train`, `X_valid`): 24 columns, unscaled, categories as integers, NaN kept. Used by Random Forest, XGBoost and LightGBM.
- **logreg tables** (`X_train_logreg`, `X_valid_logreg`): 57 columns, log-transformed, scaled and one-hot encoded. Used by the logistic regression.

In [2]:
# read one feature table from data/features, with client_id as the index
def load(name):
    return pd.read_csv(FEATURE_DIR / f"{name}.csv", index_col="client_id")

X_train, X_valid = load("X_train"), load("X_valid")
X_train_logreg, X_valid_logreg = load("X_train_logreg"), load("X_valid_logreg")
y_train, y_valid = load("y_train")["target"], load("y_valid")["target"]

# features and labels must belong to the same clients, in the same order
for X_fit, X_check in [(X_train, X_valid), (X_train_logreg, X_valid_logreg)]:
    assert (X_fit.index == y_train.index).all() and (X_check.index == y_valid.index).all()

print("train:", X_train.shape, X_train_logreg.shape, f"fraud {y_train.mean():.2%}")
print("valid:", X_valid.shape, X_valid_logreg.shape, f"fraud {y_valid.mean():.2%}")

train: (108394, 24) (108394, 57) fraud 5.58%
valid: (27099, 24) (27099, 57) fraud 5.58%


## 2. Scoring helper

The same helper as in [09_baseline-model.ipynb](09_baseline-model.ipynb), so all numbers are comparable. It takes the fraud probabilities of the validation clients.

In [3]:
def score(y_true, proba):
    # precision and recall for every possible threshold
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    # F1 for every threshold; the small number avoids dividing by zero
    f1_curve = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    # the F1-best threshold: the cut-off with the highest F1 on this data
    best = f1_curve.argmax()
    return {
        "valid ROC-AUC": roc_auc_score(y_true, proba),
        "valid PR-AUC": average_precision_score(y_true, proba),
        "precision@best": precision[best],
        "recall@best": recall[best],
        "F1@best": f1_curve[best],
    }

## 3. The four models

| Model | How it works | Weight for the fraud class |
|---|---|---|
| **Logistic regression** (baseline) | one weighted sum of the features, turned into a probability | `class_weight="balanced"` |
| **Random Forest** | many deep trees **side by side**, each on a random sample of clients and columns; their votes are averaged (bagging) | `class_weight="balanced"` |
| **XGBoost** | small trees **in a line**, each new tree fixes the mistakes of the trees before (boosting) | `scale_pos_weight` = honest / fraud clients (XGBoost has no `class_weight`) |
| **LightGBM** | also boosting, but grows each tree leaf by leaf; usually fast on big tables | `class_weight="balanced"` |

All other settings are the library defaults.

In [4]:
# ratio of honest to fraud clients in the training split, XGBoost's version of "balanced"
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = {pos_weight:.1f}")

# each model with the tables it needs: (model, training table, validation table)
models = {
    "Logistic regression (baseline)": (
        LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE),
        X_train_logreg, X_valid_logreg),
    "Random Forest": (
        RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE),
        X_train, X_valid),
    "XGBoost": (
        XGBClassifier(scale_pos_weight=pos_weight, tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE),
        X_train, X_valid),
    "LightGBM": (
        LGBMClassifier(class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE, verbose=-1),
        X_train, X_valid),
}

scale_pos_weight = 16.9


## 4. Cross-validation on the training split

Each model is trained 5 times, each time on 4/5 of the training clients and scored on the remaining 1/5. **Stratified** folds keep the 5.6% fraud share in every fold. The mean shows how good a model is; the spread (std) shows how stable the score is.

`train ROC-AUC` is the score on the clients the model was fitted on. The **gap** to the CV score is our overfitting check: a big gap means the model memorises the training clients.

Note: the scaling of the logreg tables was fitted on the whole training split in notebook 08, so the logistic regression's CV score is very slightly optimistic. For a simple scaling, this effect is negligible.

In [5]:
# the same 5 folds for every model
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_rows = {}
for name, (model, X_fit, _) in models.items():
    result = cross_validate(model, X_fit, y_train, cv=cv,
                            scoring=["roc_auc", "average_precision"], return_train_score=True)
    cv_rows[name] = {
        "CV ROC-AUC": result["test_roc_auc"].mean(),
        "CV ROC-AUC std": result["test_roc_auc"].std(),
        "CV PR-AUC": result["test_average_precision"].mean(),
        "train ROC-AUC": result["train_roc_auc"].mean(),
    }
    print(f"{name}: CV ROC-AUC {cv_rows[name]['CV ROC-AUC']:.4f}")

cv_table = pd.DataFrame(cv_rows).T
cv_table["gap"] = cv_table["train ROC-AUC"] - cv_table["CV ROC-AUC"]
cv_table.sort_values("CV ROC-AUC", ascending=False).round(4)

Logistic regression (baseline): CV ROC-AUC 0.7498


Random Forest: CV ROC-AUC 0.7376


XGBoost: CV ROC-AUC 0.7372


LightGBM: CV ROC-AUC 0.7656


,CV ROC-AUC,CV ROC-AUC std,CV PR-AUC,train ROC-AUC,gap
LightGBM,0.7656,0.0037,0.1695,0.8608,0.0951
Logistic regression (baseline),0.7498,0.0030,0.1464,0.7531,0.0033
Random Forest,0.7376,0.0051,0.1385,0.9999,0.2623
XGBoost,0.7372,0.0059,0.1516,0.9469,0.2097


## 5. Check on the validation split

Now each model is fitted once on the **whole training split** and scored on the validation clients, which no model has seen so far. If the ranking matches the cross-validation, the choice is reliable.

In [6]:
valid_rows = {}
for name, (model, X_fit, X_check) in models.items():
    # fit on all training clients, then predict the fraud probability of the validation clients
    model.fit(X_fit, y_train)
    valid_rows[name] = score(y_valid, model.predict_proba(X_check)[:, 1])

valid_table = pd.DataFrame(valid_rows).T
valid_table.sort_values("valid ROC-AUC", ascending=False).round(4)

,valid ROC-AUC,valid PR-AUC,precision@best,recall@best,F1@best
LightGBM,0.7669,0.1780,0.1641,0.4448,0.2397
Logistic regression (baseline),0.7566,0.1563,0.1542,0.4329,0.2274
XGBoost,0.7426,0.1574,0.1651,0.3562,0.2257
Random Forest,0.7412,0.1415,0.1474,0.4270,0.2191


## 6. Comparison and choice

One table with all scores, sorted by the selection rule (mean CV ROC-AUC).

In [7]:
# cross-validation and validation scores side by side, best CV ROC-AUC on top
comparison = cv_table.join(valid_table).sort_values("CV ROC-AUC", ascending=False)
comparison.round(3)

,CV ROC-AUC,CV ROC-AUC std,CV PR-AUC,train ROC-AUC,gap,valid ROC-AUC,valid PR-AUC,precision@best,recall@best,F1@best
LightGBM,0.766,0.004,0.169,0.861,0.095,0.767,0.178,0.164,0.445,0.240
Logistic regression (baseline),0.750,0.003,0.146,0.753,0.003,0.757,0.156,0.154,0.433,0.227
Random Forest,0.738,0.005,0.139,1.000,0.262,0.741,0.141,0.147,0.427,0.219
XGBoost,0.737,0.006,0.152,0.947,0.210,0.743,0.157,0.165,0.356,0.226


In [8]:
# the selection rule: highest mean CV ROC-AUC
chosen = comparison["CV ROC-AUC"].idxmax()
runner_up = comparison.index[1]
margin = comparison.loc[chosen, "CV ROC-AUC"] - comparison.loc[runner_up, "CV ROC-AUC"]

print(f"chosen model: {chosen}")
print(f"CV ROC-AUC {comparison.loc[chosen, 'CV ROC-AUC']:.4f}, "
      f"{margin:.4f} ahead of {runner_up} ({comparison.loc[runner_up, 'CV ROC-AUC']:.4f})")

# second check: is it also first on the validation split?
valid_best = comparison["valid ROC-AUC"].idxmax()
print("also best on the validation split:", "yes" if valid_best == chosen else f"no, there it is {valid_best}")

chosen model: LightGBM
CV ROC-AUC 0.7656, 0.0158 ahead of Logistic regression (baseline) (0.7498)
also best on the validation split: yes


## Result

**LightGBM is chosen for hyperparameter tuning in notebook 11.**

| Model (default settings, weighted) | CV ROC-AUC (± std) | CV PR-AUC | Gap | Valid ROC-AUC | Valid PR-AUC |
|---|---|---|---|---|---|
| **LightGBM** | **0.766** (± 0.004) | **0.169** | 0.095 | **0.767** | **0.178** |
| Logistic regression (baseline) | 0.750 (± 0.003) | 0.146 | 0.003 | 0.757 | 0.156 |
| Random Forest | 0.738 (± 0.005) | 0.139 | 0.262 | 0.741 | 0.141 |
| XGBoost | 0.737 (± 0.006) | 0.152 | 0.210 | 0.743 | 0.157 |

Why LightGBM:
- **Highest ROC-AUC**, our main metric: 0.766 in cross-validation, 0.016 ahead of the baseline. That is about four times the spread between the folds, so the lead is not luck.
- **The validation split agrees**: LightGBM is first there too (0.767), so the choice does not depend on one way of splitting.
- **Best PR-AUC** as well, so the top of its ranking is the cleanest of the four. At the F1-best threshold it finds 44% of the fraud cases with 16% precision (upper bound).
- **Moderate overfitting** (gap 0.095). Random Forest (0.262) and XGBoost (0.210) memorise the training clients much more with default settings.

Keep in mind:
- Only LightGBM clearly beats the **logistic regression baseline**. With default settings, Random Forest and XGBoost are worse than the baseline.
- The XGBoost here is weighted (`scale_pos_weight`), as decided. Tuning could change the picture for XGBoost too, but the selection rule compares default models only.

**Next step (notebook 11):** tune LightGBM with a random search on the training split (5-fold stratified CV, scoring ROC-AUC, class weight kept), then check the tuned model once on the validation split.